# Full-row causal factor + tree experiment
Development only; previously reused cohorts. Five fixed factor seeds per prior-day cutoff; matched control and treatment trees on all rows of days 3–5. Fixed 800 rounds and 10% rank blend. Day5 confirmation only after +0.0005 adjusted day6 gain, matched-control gain and ordinary nonregression. No competition CSV or official-score claim. Attach the existing causal feature and neural-history datasets; enable T4 GPU. Estimated 10–20 minutes on one T4 (about 0.17–0.33 GPU-hours; allocation accounting may differ). Save & Run All preserves the working output.


In [ ]:
from pathlib import Path
import sys,hashlib
p=Path("/kaggle/working/tied_history_gpu_bag.py")
p.write_text('"""Five predeclared tied-history GPU seeds using the existing neural bundle.\n\nIndependent full-prefix reconstruction, matched seed7201 control, fixed 5% rank\nblend. No validation early stopping, seed selection, or competition CSV export.\nThe cohorts were already reused and cannot support promotion by themselves.\n"""\nimport gc\nimport hashlib\nimport json\nimport subprocess\nimport time\nfrom bisect import bisect_left\nfrom collections import defaultdict\nfrom pathlib import Path\nimport numpy as np\nimport torch\nfrom torch import nn\nfrom scipy.stats import rankdata\nfrom sklearn.metrics import roc_auc_score\n\nRECIPE = dict(seeds=[7201, 7202, 7203, 7204, 7205], steps=2000,\n              batch_size=1024, auxiliary_batch_size=512, auxiliary_weight=0.1,\n              dimensions=32, learning_rate=0.01, sampled_vector_regularization=0.002,\n              negatives=9, context_limit=100, blend_weight=0.05,\n              minimum_primary_adjusted_gain=0.0005, cpu_threads=2)\nEXPECTED = {\n    \'events.npz\': \'3e2d628a35a9e69c6c75b0a98da28ee9dab96857209b2d16def26ce7d5c8249e\',\n    \'val5.npz\': \'6f30b7c3b825a3481006b83f02b81d617b113fb40d4898f34f504b6b9473fd3d\',\n    \'val6.npz\': \'963b8f1c8ff1edbeb88955cc087ddeccf8a4784472d488b1270b2eb6deff5d8d\',\n}\nPREPARED = {\n    6: {\n        \'fit_h\': \'91735dca1e65c475254bb60ed09cd8802f45bb9ca152104c6a277cd9e0d9ed59\',\n        \'fit_i\': \'a4fcee5ec2b954efb27dc3a6581f85fb0994d3b4494891d1859b6c65a5a4e64d\',\n        \'fit_y\': \'c3526f5dc44e90e58c25d18c16415a30ef1b144bfb21c8fd0d388535c993cacc\',\n        \'aux_h\': \'30ff4574d87fa0a88572401b851b68b84f9b4266cfbd017173542174c2dfa868\',\n        \'aux_target\': \'ca8539e97c620ebcadcfb609876788f8615d19fd813ef604266872e0bea6844e\',\n        \'val_h\': \'652d5c750a633e140f14d20a07e004c433ba145a29a3265f3d75a8bbcd4e25f2\',\n        \'val_i\': \'a6fe85b78545ee78477ae90dc02562761a34b0c5100f15ccef2ddf8c98c13019\',\n    },\n    5: {\n        \'fit_h\': \'f1c6da162f62802a6f7ab09358c096e6b54bba2d96e04b04923a0a9841251b39\',\n        \'fit_i\': \'c97faba33966617ada0873354c13208bb8e256c78e9933f6e6a77ab59a56a419\',\n        \'fit_y\': \'4b0f10af758de988b44326a64e6e5bb4672f4ae0581c0a2ecdc97242c8ee0b33\',\n        \'aux_h\': \'724f9d7a0c5b0a6407249ca53b5613d749149eed011a531161f9f1a622981a36\',\n        \'aux_target\': \'456a892b8b736f3e4ee83b76d872656c964f90491ea85b1b99285eb3bf94c994\',\n        \'val_h\': \'3819af854a71b96674f4c936d331864fbe81eda65d4b04fc861643d8444e0d05\',\n        \'val_i\': \'6915c0db249ea82079daf33e1fac4db1d677ede77156de01ec24fab0e0daaeb4\',\n    },\n}\n\n\ndef verify_prepared(prepared, day):\n    checked = {}\n    for name, expected in PREPARED[day].items():\n        view = memoryview(np.ascontiguousarray(prepared[name])).cast(\'B\')\n        value = hashlib.sha256()\n        for offset in range(0, len(view), 4*1024*1024):\n            value.update(view[offset:offset+4*1024*1024])\n        checked[name] = value.hexdigest()\n        assert checked[name] == expected, f\'CPU context mismatch: day{day} {name}\'\n    return checked\n\n\ndef digest(path):\n    h = hashlib.sha256()\n    with Path(path).open(\'rb\') as stream:\n        for block in iter(lambda: stream.read(4*1024*1024), b\'\'):\n            h.update(block)\n    return h.hexdigest()\n\n\nclass BagFactor(nn.Module):\n    def __init__(self, n_items, dimensions=32):\n        super().__init__()\n        self.items = nn.Embedding(n_items, dimensions)\n        self.bias = nn.Parameter(torch.zeros(n_items))\n        self.global_bias = nn.Parameter(torch.tensor(-2.2))\n        nn.init.normal_(self.items.weight, std=0.005)\n\n    def forward_and_penalty(self, history, candidates):\n        valid = history >= 0\n        vectors = self.items(history.clamp_min(0))*valid.unsqueeze(-1)\n        count = valid.sum(1).clamp_min(1).unsqueeze(-1)\n        profile = vectors.sum(1)/count\n        candidate_vectors = self.items(candidates)\n        bias = self.bias[candidates]\n        logits = ((profile.unsqueeze(1)*candidate_vectors).sum(-1)+bias+self.global_bias)\n        penalty = (vectors.square().sum()/valid.sum().clamp_min(1)\n                   + candidate_vectors.square().sum(-1).mean()+bias.square().mean())\n        return logits, penalty\n\n    def forward(self, history, candidates):\n        return self.forward_and_penalty(history, candidates)[0]\n\n\ndef restore_timestamps(data):\n    valid = data[\'hist\'] >= 0\n    gaps = data[\'gaps\'].astype(\'float64\')\n    if not np.isfinite(gaps).all() or (gaps < 0).any():\n        raise ValueError(\'Invalid history ages\')\n    stamps = np.rint(data[\'timestamp\'].astype(\'int64\')[:, None]-gaps*86400).astype(\'int64\')\n    if not np.all((stamps < data[\'timestamp\'][:, None]) | ~valid):\n        raise ValueError(\'Noncausal reconstructed history\')\n    return stamps\n\n\ndef pad(clicks):\n    result = np.full(RECIPE[\'context_limit\'], -1, np.int32)\n    chosen = sorted(clicks, reverse=True)[:len(result)]\n    result[:len(chosen)] = [item for _, item in chosen]\n    return result\n\n\ndef prepare(data, stamps, validation, cutoff, progress=True):\n    """Reconstruct identical CPU fitting/auxiliary contexts from raw event IDs."""\n    fit_ids = np.flatnonzero(data[\'timestamp\'] < cutoff)\n    ledger = defaultdict(set)\n    for n, q in enumerate(fit_ids):\n        user = int(data[\'users\'][q])\n        for item, timestamp in zip(data[\'hist\'][q], stamps[q]):\n            if item >= 0:\n                assert timestamp < data[\'timestamp\'][q] < cutoff\n                ledger[user].add((int(timestamp), int(item)))\n        target = int(data[\'target\'][q])\n        if target < 10:\n            ledger[user].add((int(data[\'timestamp\'][q]), int(data[\'items\'][q, target])))\n        if progress and n and n % 100000 == 0:\n            print(json.dumps(dict(stage=\'ledger\', fitting_events=n)), flush=True)\n    ledger = {user: tuple(sorted(clicks)) for user, clicks in ledger.items()}\n    def prior(user, timestamp):\n        clicks = ledger.get(user, ())\n        stop = bisect_left([t for t, _ in clicks], timestamp)\n        return clicks[max(0, stop-RECIPE[\'context_limit\']):stop]\n    fit_h = np.empty((len(fit_ids), RECIPE[\'context_limit\']), np.int32)\n    for n, q in enumerate(fit_ids):\n        fit_h[n] = pad(prior(int(data[\'users\'][q]), int(data[\'timestamp\'][q])))\n    targets, auxiliary = [], []\n    for clicks in ledger.values():\n        for target in clicks:\n            others = [value for value in clicks if value != target and value[0] < cutoff]\n            if others:\n                targets.append(target[1])\n                auxiliary.append(pad(others))\n    auxiliary = np.asarray(auxiliary, dtype=np.int32)\n    targets = np.asarray(targets, dtype=np.int32)\n    ids = validation[\'indices\'].astype(\'int64\')\n    future = np.flatnonzero((data[\'timestamp\'] >= cutoff) & (data[\'timestamp\'] < cutoff+86400))\n    _, first = np.unique(data[\'users\'][future], return_index=True)\n    assert np.array_equal(np.sort(future[first]), np.sort(ids))\n    y = (data[\'target\'][ids, None] == np.arange(10)).astype(\'float32\')\n    assert np.array_equal(y.ravel(), validation[\'y\'])\n    val_h = np.empty((len(ids), RECIPE[\'context_limit\']), np.int32)\n    for n, q in enumerate(ids):\n        clicks = set(prior(int(data[\'users\'][q]), int(data[\'timestamp\'][q])))\n        clicks.update((int(t), int(i)) for i, t in zip(data[\'hist\'][q], stamps[q]) if i >= 0)\n        val_h[n] = pad(clicks)\n    n_items = max(int(data[\'items\'].max()), int(data[\'hist\'].max()))+1\n    counts = np.bincount(data[\'items\'][fit_ids].ravel(), minlength=n_items).astype(\'float64\')\n    probability = counts/counts.sum()\n    fit_y = (data[\'target\'][fit_ids, None] == np.arange(10)).astype(\'float32\')\n    return dict(fit_h=fit_h, fit_i=data[\'items\'][fit_ids], fit_y=fit_y,\n                aux_h=auxiliary, aux_target=targets, val_h=val_h,\n                val_i=data[\'items\'][ids], exposure_probability=probability, n_items=n_items)\n\n\ndef load(root):\n    for name, expected in EXPECTED.items():\n        assert digest(root/name) == expected, f\'Unexpected bundle version: {name}\'\n    with np.load(root/\'events.npz\') as z:\n        data = {key: z[key].copy() for key in z.files}\n    assert np.isin(data[\'target\'], np.arange(11)).all()\n    assert data[\'items\'].shape == (len(data[\'timestamp\']), 10)\n    assert np.all(np.diff(data[\'timestamp\'].astype(\'int64\')) >= 0)\n    stamps = restore_timestamps(data)\n    return data, stamps\n\n\ndef one_seed(prepared, seed, output, device):\n    torch.manual_seed(seed)\n    torch.cuda.manual_seed_all(seed)\n    rng = np.random.default_rng(seed)\n    model = BagFactor(prepared[\'n_items\'], RECIPE[\'dimensions\']).to(device)\n    with torch.no_grad():\n        rate = float(prepared[\'fit_y\'].mean().item())\n        model.global_bias.fill_(np.log(rate/(1-rate)))\n    optimizer = torch.optim.Adam(model.parameters(), lr=RECIPE[\'learning_rate\'])\n    started = time.monotonic()\n    for step in range(RECIPE[\'steps\']):\n        ii = rng.integers(len(prepared[\'fit_h\']), size=RECIPE[\'batch_size\'])\n        ai = rng.integers(len(prepared[\'aux_h\']), size=RECIPE[\'auxiliary_batch_size\'])\n        ids = torch.as_tensor(ii, device=device, dtype=torch.long)\n        aids = torch.as_tensor(ai, device=device, dtype=torch.long)\n        history = prepared[\'fit_h\'][ids].long()\n        candidates = prepared[\'fit_i\'][ids].long()\n        negatives = rng.choice(prepared[\'n_items\'], size=(len(ai), RECIPE[\'negatives\']),\n                               p=prepared[\'exposure_probability\'])\n        negative_ids = torch.as_tensor(negatives, device=device, dtype=torch.long)\n        aux_candidates = torch.cat([prepared[\'aux_target\'][aids, None].long(), negative_ids], dim=1)\n        aux_history = prepared[\'aux_h\'][aids].long()\n        optimizer.zero_grad()\n        logits, penalty = model.forward_and_penalty(history, candidates)\n        aux_logits, aux_penalty = model.forward_and_penalty(aux_history, aux_candidates)\n        aux_labels = torch.zeros_like(aux_logits)\n        aux_labels[:, 0] = 1\n        loss = (nn.functional.binary_cross_entropy_with_logits(logits, prepared[\'fit_y\'][ids])\n                + RECIPE[\'auxiliary_weight\']*nn.functional.binary_cross_entropy_with_logits(aux_logits, aux_labels)\n                + RECIPE[\'sampled_vector_regularization\']*(penalty+RECIPE[\'auxiliary_weight\']*aux_penalty))\n        if not torch.isfinite(loss):\n            raise FloatingPointError(\'Nonfinite loss\')\n        loss.backward()\n        optimizer.step()\n        if (step+1) % 200 == 0:\n            print(json.dumps(dict(seed=seed, step=step+1, loss=float(loss.detach()),\n                                  elapsed_seconds=time.monotonic()-started)), flush=True)\n    def predict(member):\n        result = []\n        with torch.no_grad():\n            for offset in range(0, len(prepared[\'val_h\']), 1024):\n                h = prepared[\'val_h\'][offset:offset+1024].long()\n                i = prepared[\'val_i\'][offset:offset+1024].long()\n                result.append(member(h, i).sigmoid().cpu().numpy().ravel())\n        return np.concatenate(result)\n    pred = predict(model)\n    state = {key: value.detach().cpu() for key, value in model.state_dict().items()}\n    torch.save(state, output/f\'seed{seed}.pt\')\n    replay = BagFactor(prepared[\'n_items\'], RECIPE[\'dimensions\']).to(device)\n    replay.load_state_dict(torch.load(output/f\'seed{seed}.pt\', weights_only=True))\n    assert np.array_equal(pred, predict(replay))\n    np.save(output/f\'seed{seed}_pred.npy\', pred)\n    print(json.dumps(dict(seed=seed, checkpoint_replay=\'exact_all_rows\',\n                          elapsed_seconds=time.monotonic()-started)), flush=True)\n    del model, replay, optimizer\n    gc.collect()\n    torch.cuda.empty_cache()\n    return pred\n\n\ndef run(root, output):\n    root, output = Path(root), Path(output)\n    assert torch.cuda.is_available(), \'GPU required; refusing a long CPU fallback\'\n    output.mkdir(parents=True, exist_ok=False)\n    torch.set_num_threads(RECIPE[\'cpu_threads\'])\n    torch.backends.cuda.matmul.allow_tf32 = False\n    device = torch.device(\'cuda:0\')\n    started = time.monotonic()\n    gpu = dict(cuda_available=True, name=torch.cuda.get_device_name(0),\n               torch_version=torch.__version__, memory_bytes=torch.cuda.get_device_properties(0).total_memory,\n               nvidia_smi=subprocess.check_output([\'nvidia-smi\', \'--query-gpu=name,driver_version\', \'--format=csv,noheader\'], text=True))\n    frozen = dict(recipe=RECIPE, gpu=gpu, source_sha256=digest(__file__),\n                  inputs=EXPECTED, fresh_holdout=False, official_auc=None,\n                  note=\'Fixed five seeds, matched single-seed GPU control; no seed selection or per-seed validation scoring\')\n    (output/\'frozen.json\').write_text(json.dumps(frozen, indent=2))\n    print(json.dumps(frozen), flush=True)\n    data, stamps = load(root)\n    reports = []\n    for day in [6, 5]:\n        day_out = output/f\'day{day}\'\n        day_out.mkdir()\n        with np.load(root/f\'val{day}.npz\') as z:\n            validation = {key: z[key].copy() for key in z.files}\n        cutoff = int(data[\'timestamp\'][0]) + day*86400\n        prepared = prepare(data, stamps, validation, cutoff)\n        hashes = verify_prepared(prepared, day)\n        (day_out/\'prepared_array_hashes.json\').write_text(json.dumps(hashes, indent=2))\n        print(json.dumps(dict(stage=\'prepared\', day=day, fitting_events=len(prepared[\'fit_h\']),\n                              auxiliary_clicks=len(prepared[\'aux_h\']))), flush=True)\n        for name in [\'val_h\', \'val_i\']:\n            np.save(day_out/f\'{name}.npy\', prepared[name])\n        for name in [\'fit_h\', \'fit_i\', \'fit_y\', \'aux_h\', \'aux_target\', \'val_h\', \'val_i\']:\n            prepared[name] = torch.as_tensor(prepared[name], device=device)\n        predictions = [one_seed(prepared, seed, day_out, device) for seed in RECIPE[\'seeds\']]\n        control_raw = predictions[0]\n        bag_raw = np.mean(np.stack(predictions).astype(\'float64\'), axis=0)\n        rank = lambda x: rankdata(x, method=\'average\')/(len(x)+1)\n        blend = lambda p: (1-RECIPE[\'blend_weight\'])*rank(validation[\'baseline\']) + RECIPE[\'blend_weight\']*rank(p)\n        control, bag = blend(control_raw), blend(bag_raw)\n        def metrics(values):\n            return dict(ordinary_auc=float(roc_auc_score(validation[\'y\'], values)),\n                        adjusted_auc=float(roc_auc_score(validation[\'y\'], values, sample_weight=validation[\'weights\'])))\n        record = dict(day=day, baseline=metrics(validation[\'baseline\']), control=metrics(control),\n                      bag=metrics(bag), raw_control=metrics(control_raw), raw_bag=metrics(bag_raw),\n                      all_five_checkpoint_replays=\'exact_all_rows\', promoted=False)\n        record[\'adjusted_gain\'] = record[\'bag\'][\'adjusted_auc\']-record[\'baseline\'][\'adjusted_auc\']\n        record[\'adjusted_gain_vs_control\'] = record[\'bag\'][\'adjusted_auc\']-record[\'control\'][\'adjusted_auc\']\n        record[\'development_gate\'] = (record[\'adjusted_gain\'] >= RECIPE[\'minimum_primary_adjusted_gain\']\n                                      and record[\'adjusted_gain_vs_control\'] > 0\n                                      and record[\'bag\'][\'ordinary_auc\'] >= record[\'baseline\'][\'ordinary_auc\'])\n        np.savez_compressed(day_out/\'predictions.npz\', y=validation[\'y\'], events=np.repeat(validation[\'indices\'], 10),\n                            weights=validation[\'weights\'], baseline=validation[\'baseline\'],\n                            control_raw=control_raw, bag_raw=bag_raw, control=control, bag=bag)\n        (day_out/\'report.json\').write_text(json.dumps(record, indent=2))\n        reports.append(record)\n        print(json.dumps(record), flush=True)\n        del prepared, predictions\n        gc.collect()\n        torch.cuda.empty_cache()\n        if not record[\'development_gate\']:\n            break\n    summary = dict(reports=reports, day5_run=len(reports)==2, target_achieved=False,\n                   local_adjusted_target_reached=reports[0][\'bag\'][\'adjusted_auc\'] >= 0.83,\n                   promoted=False, csv_ready=False, official_auc=None, fresh_holdout=False,\n                   elapsed_seconds=time.monotonic()-started)\n    (output/\'summary.json\').write_text(json.dumps(summary, indent=2))\n    print(json.dumps(summary), flush=True)\n    return summary\n',encoding="utf-8")
assert hashlib.sha256(p.read_bytes()).hexdigest()=='02b53689d03e493f76845f29837150cc9d5d2372be6910b5b44145f5468298e1'
p=Path("/kaggle/working/tied_history_oof_tree.py")
p.write_text('"""Full-row temporal OOF factor features with a matched pointwise tree control.\n\nEach day\'s factors fit strictly earlier events. Tree arms share rows, weights\nand settings. Reused validation remains exploratory; no submission is exported.\n"""\nimport gc\nimport importlib.util\nimport json\nimport time\nfrom collections import defaultdict\nfrom pathlib import Path\nimport numpy as np\nimport torch\nimport xgboost as xgb\nfrom scipy.special import logit, softmax\nfrom scipy.stats import rankdata\nfrom sklearn.metrics import roc_auc_score\nimport tied_history_gpu_bag as bag\n\nROUNDS = 800\nPARAMS = dict(objective=\'binary:logistic\', tree_method=\'hist\', device=\'cuda:0\',\n              max_bin=63, max_depth=6, eta=.05, min_child_weight=50,\n              subsample=.8, colsample_bytree=.9, reg_lambda=5., reg_alpha=.1,\n              nthread=2, seed=7501, max_cached_hist_node=32)\nBLEND = .1\nNAMES = [\'factor_probability\', \'factor_logit\', \'seed_probability_std\',\n         \'mean_dot\', \'cosine\', \'profile_norm\', \'candidate_norm\', \'candidate_bias\',\n         \'maximum_history_dot\', \'history_dot_std\', \'log_full_history_count\',\n         \'slate_centered_logit\', \'slate_max_relative_logit\', \'slate_logit_std\',\n         \'slate_probability_sum\', \'slate_softmax\', \'factor_percentile\']\n\n\ndef module(name, path):\n    spec = importlib.util.spec_from_file_location(name, path)\n    result = importlib.util.module_from_spec(spec)\n    spec.loader.exec_module(result)\n    return result\n\n\ndef first_ids(data, cutoff):\n    ids = np.flatnonzero((data[\'timestamp\'] >= cutoff) & (data[\'timestamp\'] < cutoff+86400))\n    _, first = np.unique(data[\'users\'][ids], return_index=True)\n    return ids[first]\n\n\ndef score_contexts(data, stamps, cutoff, ids):\n    """Only prefix labels and each scored event\'s supplied prior history."""\n    ledger = defaultdict(set)\n    for q in np.flatnonzero(data[\'timestamp\'] < cutoff):\n        user = int(data[\'users\'][q])\n        for item, stamp in zip(data[\'hist\'][q], stamps[q]):\n            if item >= 0:\n                assert stamp < data[\'timestamp\'][q] < cutoff\n                ledger[user].add((int(stamp), int(item)))\n        target = int(data[\'target\'][q])\n        if target < 10:\n            ledger[user].add((int(data[\'timestamp\'][q]), int(data[\'items\'][q, target])))\n    ledger = {user: tuple(sorted(values, reverse=True)[:100]) for user, values in ledger.items()}\n    contexts = np.full((len(ids), 100), -1, np.int32)\n    for n, q in enumerate(ids):\n        assert data[\'timestamp\'][q] >= cutoff\n        values = set(ledger.get(int(data[\'users\'][q]), ()))\n        for item, stamp in zip(data[\'hist\'][q], stamps[q]):\n            if item >= 0:\n                assert stamp < data[\'timestamp\'][q]\n                values.add((int(stamp), int(item)))\n        contexts[n] = bag.pad(values)\n    return contexts\n\n\ndef geometry(model, history, items):\n    valid = history >= 0\n    vectors = model.items(history.clamp_min(0))*valid.unsqueeze(-1)\n    count = valid.sum(1).clamp_min(1)\n    profile = vectors.sum(1)/count[:, None]\n    candidates = model.items(items)\n    dot = (profile[:, None, :]*candidates).sum(-1)\n    pn = torch.linalg.vector_norm(profile, dim=-1)\n    cn = torch.linalg.vector_norm(candidates, dim=-1)\n    cosine = dot/(pn[:, None]*cn).clamp_min(1e-9)\n    pair = torch.einsum(\'bhd,bcd->bch\', vectors, candidates)\n    maximum = pair.masked_fill(~valid[:, None, :], -1e30).max(2).values\n    maximum[~valid.any(1)] = 0\n    variance = ((pair*pair).sum(2)/count[:, None]-dot*dot).clamp_min(0)\n    return torch.stack([dot, cosine, pn[:, None].expand_as(dot), cn,\n                        model.bias[items], maximum, variance.sqrt()], dim=-1).reshape(-1, 7)\n\n\ndef extra_features(probabilities, mean_geometry, history):\n    all_p = np.stack(probabilities).astype(\'float64\')\n    probability = all_p.mean(0)\n    logits = logit(np.clip(probability, 1e-6, 1-1e-6)).reshape(-1, 10)\n    p = probability.reshape(-1, 10)\n    values = [probability, logits.ravel(), all_p.std(0)]\n    values.extend(mean_geometry[:, i] for i in range(7))\n    values.extend([np.repeat(np.log1p((history >= 0).sum(1)), 10),\n                   (logits-logits.mean(1, keepdims=True)).ravel(),\n                   (logits-logits.max(1, keepdims=True)).ravel(),\n                   np.repeat(logits.std(1), 10), np.repeat(p.sum(1), 10),\n                   softmax(logits, axis=1).ravel(),\n                   rankdata(probability, method=\'average\')/(len(probability)+1)])\n    result = np.column_stack(values).astype(\'float32\')\n    assert result.shape[1] == len(NAMES) and np.isfinite(result).all()\n    return result\n\n\ndef build_day(data, stamps, neural_root, output, day):\n    output.mkdir()\n    cutoff = int(data[\'timestamp\'][0])+day*86400\n    first = first_ids(data, cutoff)\n    validation = dict(indices=first, y=(data[\'target\'][first, None] == np.arange(10)).astype(\'float32\').ravel())\n    if day in [5, 6]:\n        with np.load(neural_root/f\'val{day}.npz\') as original:\n            assert np.array_equal(first, original[\'indices\'])\n            assert np.array_equal(validation[\'y\'], original[\'y\'])\n    prepared = bag.prepare(data, stamps, validation, cutoff)\n    if day in [5, 6]:\n        checked = bag.verify_prepared(prepared, day)\n        (output/\'prepared_hashes.json\').write_text(json.dumps(checked, indent=2))\n    ids = first if day == 6 else np.flatnonzero((data[\'timestamp\'] >= cutoff) & (data[\'timestamp\'] < cutoff+86400))\n    history = score_contexts(data, stamps, cutoff, ids)\n    # Confirm scoring contexts on the first-event subset equal the prior audit.\n    if day == 6:\n        assert np.array_equal(history, prepared[\'val_h\'])\n    else:\n        assert np.array_equal(history[np.searchsorted(ids, first)], prepared[\'val_h\'])\n    prepared[\'val_h\'], prepared[\'val_i\'] = history, data[\'items\'][ids]\n    for name in [\'fit_h\', \'fit_i\', \'fit_y\', \'aux_h\', \'aux_target\', \'val_h\', \'val_i\']:\n        prepared[name] = torch.as_tensor(prepared[name], device=\'cuda:0\')\n    predictions, mean_geometry = [], None\n    for seed in bag.RECIPE[\'seeds\']:\n        predictions.append(bag.one_seed(prepared, seed, output, torch.device(\'cuda:0\')))\n        model = bag.BagFactor(prepared[\'n_items\'], bag.RECIPE[\'dimensions\']).cuda()\n        model.load_state_dict(torch.load(output/f\'seed{seed}.pt\', weights_only=True))\n        pieces = []\n        with torch.no_grad():\n            for off in range(0, len(ids), 512):\n                pieces.append(geometry(model, prepared[\'val_h\'][off:off+512].long(),\n                                       prepared[\'val_i\'][off:off+512].long()).cpu().numpy())\n        values = np.concatenate(pieces).astype(\'float64\')\n        mean_geometry = values if mean_geometry is None else mean_geometry+values\n        del model, pieces, values\n        gc.collect()\n        torch.cuda.empty_cache()\n    extra = extra_features(predictions, mean_geometry/len(predictions), history)\n    np.save(output/\'features.npy\', extra)\n    np.save(output/\'event_ids.npy\', ids)\n    labels = (data[\'target\'][ids, None] == np.arange(10)).astype(\'float32\').ravel()\n    np.save(output/\'labels.npy\', labels)\n    report = dict(day=day, factor_fit_last_timestamp=int(data[\'timestamp\'][data[\'timestamp\'] < cutoff].max()),\n                  cutoff=cutoff, score_first_timestamp=int(data[\'timestamp\'][ids].min()),\n                  scored_events=len(ids), feature_count=extra.shape[1], feature_sha256=bag.digest(output/\'features.npy\'),\n                  checkpoint_replay=\'all_five_exact_all_scored_rows\', labels_used_for_factor_fit=\'strict prefix only\')\n    (output/\'report.json\').write_text(json.dumps(report, indent=2))\n    print(json.dumps(report), flush=True)\n    del prepared, predictions, mean_geometry\n    gc.collect()\n    torch.cuda.empty_cache()\n\n\nclass Augmented:\n    def __init__(self, base, paths):\n        self.base = base\n        self.parts = [np.load(path, mmap_mode=\'r\') for path in paths]\n        assert self.parts and all(part.ndim == 2 and part.shape[1] == len(NAMES) for part in self.parts)\n        self.ends = np.cumsum([len(part) for part in self.parts])\n        self.starts = np.r_[0, self.ends[:-1]]\n        assert len(base) == int(self.ends[-1])\n\n    def __len__(self):\n        return len(self.base)\n\n    def block(self, start, end):\n        assert 0 <= start < end <= len(self)\n        pieces = []\n        for left, right, part in zip(self.starts, self.ends, self.parts):\n            a, b = max(start, int(left)), min(end, int(right))\n            if b > a:\n                pieces.append(np.asarray(part[a-int(left):b-int(left)]))\n        return np.column_stack([self.base.block(start, end), np.concatenate(pieces)])\n\n\nclass Iterator(xgb.DataIter):\n    def __init__(self, features, labels, weights, cp):\n        super().__init__(release_data=True)\n        self.features, self.labels, self.weights, self.cp = features, labels, weights, cp\n        self.position = 0\n\n    def reset(self):\n        self.position = 0\n\n    def next(self, input_data):\n        if self.position >= len(self.labels):\n            return False\n        stop = min(self.position+100000, len(self.labels))\n        start = self.position\n        input_data(data=self.cp.asarray(self.features.block(start, stop)),\n                   label=self.cp.asarray(self.labels[start:stop]),\n                   weight=self.cp.asarray(self.weights[start:stop]))\n        self.position = stop\n        return True\n\n\nclass Progress(xgb.callback.TrainingCallback):\n    def after_iteration(self, model, epoch, evals_log):\n        if (epoch+1) % 100 == 0:\n            print(json.dumps(dict(stage=\'tree_training\', round=epoch+1, total=ROUNDS)), flush=True)\n        return False\n\n\ndef target_share(data, neural_root):\n    with np.load(neural_root/\'val6.npz\') as z:\n        groups = np.digitize((data[\'hist\'][z[\'indices\']] >= 0).sum(1), [0, 4, 9, 19], right=True)\n        source = np.bincount(groups, minlength=5)/len(groups)\n        weights = z[\'weights\'].reshape(-1, 10)[:, 0]\n        ratio = np.array([weights[groups == group][0] for group in range(5)])\n        assert np.allclose(weights, ratio[groups], rtol=1e-12, atol=1e-12)\n        result = source*ratio\n        assert np.isclose(result.sum(), 1)\n        return result\n\n\ndef fit_trees(root, neural_root, output, data, manifest, helpers, day):\n    import cupy as cp\n    output.mkdir()\n    days = list(range(day-3, day))\n    specs = [manifest[\'train\'][str(d)] for d in days]\n    labels, ids = [], []\n    start = int(data[\'timestamp\'][0])\n    for d, spec in zip(days, specs):\n        meta = json.loads((root/spec[\'metadata\']).read_text())\n        assert meta[\'last_timestamp\'] < start+day*86400 and meta[\'feature_cutoff\'] == start+d*86400\n        event_ids = np.load(output.parent/f\'factors{d}/event_ids.npy\')\n        y = np.load(root/spec[\'labels\'])\n        assert np.array_equal(y, (data[\'target\'][event_ids, None] == np.arange(10)).ravel())\n        labels.append(y)\n        ids.append(event_ids)\n    labels, ids = np.concatenate(labels).astype(\'float32\'), np.concatenate(ids)\n    groups = np.digitize((data[\'hist\'][ids] >= 0).sum(1), [0, 4, 9, 19], right=True)\n    shares = target_share(data, neural_root)\n    source = np.bincount(groups, minlength=5)/len(groups)\n    assert (source > 0).all()\n    weights = np.repeat((shares/source)[groups], 10).astype(\'float32\')\n    weights /= weights.mean()\n    base = helpers.FeatureBlocks(root, [spec[\'features\'] for spec in specs])\n    treatment = Augmented(base, [output.parent/f\'factors{d}/features.npy\' for d in days])\n    validation = manifest[\'validation\'][str(day)]\n    val = np.load(root/validation[\'data\'])\n    val_base = helpers.FeatureBlocks(root, [validation[\'features\']])\n    score_ids = np.load(output.parent/f\'factors{day}/event_ids.npy\')\n    desired = val[\'events\'][::10].astype(\'int64\')\n    if day == 6:\n        assert np.array_equal(desired, score_ids)\n        extra = np.load(output.parent/f\'factors{day}/features.npy\', mmap_mode=\'r\')\n    else:\n        positions = np.searchsorted(score_ids, desired)\n        assert np.array_equal(score_ids[positions], desired)\n        extra = np.load(output.parent/f\'factors{day}/features.npy\', mmap_mode=\'r\').reshape(-1, 10, len(NAMES))[positions].reshape(-1, len(NAMES))\n    extra_path = output/\'validation_extra.npy\'\n    np.save(extra_path, extra)\n    val_augmented = Augmented(val_base, [extra_path])\n    predictions = {}\n    for arm, features, future in [(\'control\', base, val_base), (\'treatment\', treatment, val_augmented)]:\n        iterator = Iterator(features, labels, weights, cp)\n        matrix = xgb.QuantileDMatrix(iterator, nthread=2, max_bin=63)\n        assert matrix.num_row() == len(labels)\n        model = xgb.train(PARAMS, matrix, num_boost_round=ROUNDS, callbacks=[Progress()])\n        assert model.num_boosted_rounds() == ROUNDS\n        model.save_model(output/f\'{arm}.ubj\')\n        del matrix, iterator\n        gc.collect()\n        cp.get_default_memory_pool().free_all_blocks()\n        def predict(member):\n            values = []\n            for off in range(0, len(future), 25000):\n                block = cp.asarray(future.block(off, min(off+25000, len(future))))\n                values.append(cp.asnumpy(member.inplace_predict(block)))\n            return np.concatenate(values)\n        raw = predict(model)\n        replay = xgb.Booster()\n        replay.load_model(output/f\'{arm}.ubj\')\n        replay.set_param({\'device\': \'cuda:0\', \'nthread\': 2})\n        assert np.array_equal(raw, predict(replay))\n        predictions[arm+\'_raw\'] = raw\n        rank = lambda p: rankdata(np.asarray(p, dtype=\'float64\'), method=\'average\')/(len(p)+1)\n        predictions[arm] = (1-BLEND)*rank(val[\'baseline\'])+BLEND*rank(raw)\n        print(json.dumps(dict(stage=\'tree_complete\', day=day, arm=arm, model_replay=\'exact_all_rows\')), flush=True)\n        del model, replay\n        gc.collect()\n        cp.get_default_memory_pool().free_all_blocks()\n    def metric(p):\n        return dict(ordinary_auc=float(roc_auc_score(val[\'y\'], p)),\n                    adjusted_auc=float(roc_auc_score(val[\'y\'], p, sample_weight=val[\'weights\'])))\n    report = dict(day=day, fitting_days=days, fitting_rows=len(labels), baseline=metric(val[\'baseline\']),\n                  control=metric(predictions[\'control\']), treatment=metric(predictions[\'treatment\']))\n    report[\'adjusted_gain\'] = report[\'treatment\'][\'adjusted_auc\']-report[\'baseline\'][\'adjusted_auc\']\n    report[\'gain_vs_control\'] = report[\'treatment\'][\'adjusted_auc\']-report[\'control\'][\'adjusted_auc\']\n    report[\'ordinary_nonregression\'] = report[\'treatment\'][\'ordinary_auc\'] >= report[\'baseline\'][\'ordinary_auc\']\n    report[\'development_gate\'] = (report[\'adjusted_gain\'] >= .0005 and report[\'gain_vs_control\'] > 0\n                                  and report[\'ordinary_nonregression\'])\n    np.savez_compressed(output/\'predictions.npz\', y=val[\'y\'], events=val[\'events\'],\n                        weights=val[\'weights\'], baseline=val[\'baseline\'], **predictions)\n    (output/\'report.json\').write_text(json.dumps(report, indent=2))\n    print(json.dumps(report), flush=True)\n    return report\n\n\ndef run(feature_root, neural_root, output):\n    feature_root, neural_root, output = map(Path, [feature_root, neural_root, output])\n    assert torch.cuda.is_available(), \'GPU required; no CPU fallback\'\n    output.mkdir(exist_ok=False)\n    torch.set_num_threads(2)\n    torch.backends.cuda.matmul.allow_tf32 = False\n    started = time.monotonic()\n    frozen = dict(factor_recipe=bag.RECIPE, tree_params=PARAMS, rounds=ROUNDS, blend=BLEND,\n                  feature_names=NAMES, source_sha256=bag.digest(__file__),\n                  factor_source_sha256=bag.digest(bag.__file__),\n                  feature_manifest_sha256=bag.digest(feature_root/\'bundle_manifest.json\'),\n                  gpu=torch.cuda.get_device_name(0), cuda_available=True,\n                  fresh_holdout=False, promoted=False, target_achieved=False,\n                  scope=\'Full-row temporal OOF, not the previous first-event meta stack; no validation tuning\')\n    (output/\'frozen.json\').write_text(json.dumps(frozen, indent=2))\n    print(json.dumps(frozen), flush=True)\n    helpers = module(\'oof_bundle_helpers\', feature_root/\'runner.py\')\n    manifest = helpers.load_bundle(feature_root, verify=True)\n    data, stamps = bag.load(neural_root)\n    for day in [3, 4, 5, 6]:\n        build_day(data, stamps, neural_root, output/f\'factors{day}\', day)\n    reports = [fit_trees(feature_root, neural_root, output/\'trees6\', data, manifest, helpers, 6)]\n    if reports[0][\'development_gate\']:\n        build_day(data, stamps, neural_root, output/\'factors2\', 2)\n        reports.append(fit_trees(feature_root, neural_root, output/\'trees5\', data, manifest, helpers, 5))\n    result = dict(reports=reports, day5_run=len(reports)==2, promoted=False, csv_ready=False,\n                  fresh_holdout=False, official_auc=None, target_achieved=False,\n                  elapsed_seconds=time.monotonic()-started)\n    (output/\'summary.json\').write_text(json.dumps(result, indent=2))\n    print(json.dumps(result), flush=True)\n    return result\n',encoding="utf-8")
assert hashlib.sha256(p.read_bytes()).hexdigest()=='1cbd6cf658889e5daff13741d0757f8219cc280ffa83369488145f20c797ae1b'
sys.path.insert(0,"/kaggle/working")
import tied_history_oof_tree as experiment


In [ ]:
import json,subprocess
print(subprocess.run(['nvidia-smi'],capture_output=True,text=True).stdout)
roots=list(Path('/kaggle/input').rglob('bundle_manifest.json'))
neural=list(Path('/kaggle/input').rglob('neural_manifest.json'))
assert len(roots)==len(neural)==1,(roots,neural)
output=Path('/kaggle/working/tied-history-oof-tree-001')
result=experiment.run(roots[0].parent,neural[0].parent,output)
